In [8]:
# 直接使用基因名进行富集与聚类，并将结果保存到CSV
# 说明：
# - 不做任何ID与基因名的映射，NPY文件中已是基因名
# - 若无显著富集或无法聚类，则将所有基因的 Classification ID 设为 -1
# - 输出CSV包含：Gene Name, Classification ID, Cluster_Top_Terms（每类的Top术语合并为字符串）

import gseapy as gp
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

# 【重要参数设置】
NPY_FILE_PATH = 'select_genes/cSCC_Selected_Genes.npy'
OUTPUT_CSV_PATH = 'select_genes/cSCC_mapping.csv'
K_CLUSTERS = 10
SIGNIFICANCE_THRESHOLD = 0.05
TOP_TERMS_TO_REPORT = 3

def characterize_cluster(gene_list, gene_set_name='GO_Biological_Process_2023'):
    """对某个分类组的基因运行富集，返回Top术语列表（按调整P值过滤与排序）"""
    if not gene_list:
        return []
    try:
        res = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set_name,
            organism='Human',
            outdir=None,
            no_plot=True
        )
        df_char = res.results
        df_sig = df_char[df_char['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD]
        return df_sig.head(TOP_TERMS_TO_REPORT)['Term'].tolist()
    except Exception as e:
        return [f"分析失败({e})"]

# 1) 读取基因名列表（不做映射）
try:
    gene_array = np.load(NPY_FILE_PATH, allow_pickle=True)
    gene_list = [str(g).strip() for g in gene_array.flatten() if str(g).strip()]
except Exception as e:
    raise RuntimeError(f"读取NPY失败：{e}")

print(f"共读取 {len(gene_list)} 个基因名。")

# 为了输出全量结果，构造基础表（保持原顺序并去重）
seen = set()
base_genes = []
for g in gene_list:
    if g not in seen:
        seen.add(g)
        base_genes.append(g)
base_df = pd.DataFrame({'Gene Name': base_genes})

# 2) 主富集分析
go_results = gp.enrichr(
    gene_list=base_genes,
    gene_sets='GO_Biological_Process_2023',
    organism='Human',
    outdir=None,
    no_plot=True
)
df_enrichment = go_results.results
df_significant = df_enrichment[
    df_enrichment['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD
].sort_values(by='Adjusted P-value')

# 若无显著富集：全部标记为 -1 并保存
if df_significant.empty:
    print("无显著富集结果，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# 3) 构建基因-功能矩阵
all_genes = sorted(list(set(g for term_genes in df_significant['Genes'] for g in term_genes.split(';'))))
all_terms = df_significant['Term'].tolist()

gene_term_matrix = pd.DataFrame(0, index=all_genes, columns=all_terms)
for term in all_terms:
    genes_in_term = df_significant.loc[df_significant['Term'] == term, 'Genes'].iloc[0].split(';')
    # 将该术语涉及到的基因位置置为1
    gene_term_matrix.loc[[g for g in genes_in_term if g in gene_term_matrix.index], term] = 1

X = gene_term_matrix.values

# 若基因数量不足以聚类：全部标记为 -1 并保存
if len(all_genes) < K_CLUSTERS:
    print(f"可用于聚类的基因数({len(all_genes)}) < K({K_CLUSTERS})，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# 4) KMeans 聚类
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X)

df_classification = pd.DataFrame({
    'Gene Name': all_genes,
    'Classification ID': cluster_labels
}).sort_values(by='Classification ID')

# 5) 为每个分类组提取Top术语，并合并到每个基因
classification_reasons = {}
for cid in range(K_CLUSTERS):
    genes_in_class = df_classification[df_classification['Classification ID'] == cid]['Gene Name'].tolist()
    top_terms = characterize_cluster(genes_in_class)
    classification_reasons[cid] = " | ".join(top_terms) if top_terms else ""

df_classification['Cluster_Top_Terms'] = df_classification['Classification ID'].map(classification_reasons)

# 6) 与基础表合并（保证所有输入基因都在输出中）
df_out = base_df.merge(df_classification, on='Gene Name', how='left')

# 对未参与显著富集或未聚到类的基因，分类设为 -1、术语为空
df_out['Classification ID'] = df_out['Classification ID'].fillna(-1).astype(int)
df_out['Cluster_Top_Terms'] = df_out['Cluster_Top_Terms'].fillna('')

# 保存CSV
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"已保存分类结果到：{OUTPUT_CSV_PATH}")

共读取 250 个基因名。


已保存分类结果到：select_genes/cSCC_mapping.csv


In [7]:
import gseapy as gp
import pandas as pd
import numpy as np
import os
import re
from sklearn.cluster import KMeans

# --- 【重要参数设置】 ---
NPY_FILE_PATH = 'select_genes/STNet_select_genes.npy'
K_CLUSTERS = 10                   # 设定的互斥分组数量
SIGNIFICANCE_THRESHOLD = 0.05     # GO 富集分析的 P 值阈值
TOP_TERMS_TO_REPORT = 3           # 每个分类组输出的最具代表性的功能数量

# 映射表路径（包含列：HGNC ID、Approved symbol、Approved name、Ensembl ID）
MAPPING_TSV_PATH = "select_genes/STNet.tsv"

# 输出CSV路径（映射+分类合并结果）
OUTPUT_CSV_PATH = "select_genes/STNet_mapping.csv"

# --- 辅助函数：对单个基因分类组进行功能富集，以找到分类原因 ---
def characterize_cluster(gene_list, cluster_id, gene_set_name='GO_Biological_Process_2023'):
    """运行富集分析，找出该基因列表最显著的功能特征"""
    if not gene_list:
        return []
    try:
        char_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set_name,
            organism='Human',
            outdir=None,
            no_plot=True 
        )
        df_char = char_results.results
        df_sig = df_char[df_char['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD]
        top_terms = df_sig.head(TOP_TERMS_TO_REPORT)['Term'].tolist()
        return top_terms
    except Exception as e:
        return [f"分析失败 ({e})"]

# -------------------------------------------------------------------------------------
# 1) 从 .npy 文件读取基因ID（原始列表）
# -------------------------------------------------------------------------------------
try:
    gene_array = np.load(NPY_FILE_PATH, allow_pickle=True)
    raw_gene_list = [str(g).strip() for g in gene_array.flatten() if str(g).strip()]
except Exception as e:
    print(f"⚠️ 警告：读取 {NPY_FILE_PATH} 失败（{e}）。")
    raw_gene_list = []
print(f"--- 初始基因条目数：{len(raw_gene_list)} ---")

# -------------------------------------------------------------------------------------
# 2) ID->基因名映射（记录映射明细），随后仅用“命中映射”的基因名进行富集
# -------------------------------------------------------------------------------------
def normalize_ensg(x):
    s = str(x).strip()
    if not s or s.startswith("__"):  # 去除模糊/技术条目
        return None
    s = s.split('.')[0]              # 去版本号：ENSG00000121410.6 -> ENSG00000121410
    m = re.search(r'(ENSG\d+)', s)   # 提取 ENSG 主体
    return m.group(1) if m else None

try:
    # 读取映射表（使用固定列名）
    df_map = pd.read_csv(MAPPING_TSV_PATH, sep='\t', dtype=str)
    df_map = df_map[['Ensembl ID', 'Approved symbol']].dropna()
    df_map['Ensembl ID'] = df_map['Ensembl ID'].str.strip().str.split('.').str[0]
    ensg2symbol = dict(zip(df_map['Ensembl ID'], df_map['Approved symbol']))

    # 生成映射记录
    mapping_records = []
    for raw in raw_gene_list:
        norm = normalize_ensg(raw)
        sym = ensg2symbol.get(norm) if norm else None
        mapping_records.append({
            "原始ID": raw,
            "归一化ENSG": norm if norm else "",
            "Approved_symbol": sym if sym else "",
            "是否命中映射": bool(sym)
        })
    df_mapping = pd.DataFrame(mapping_records)

    # 富集所用的 gene_list：仅使用“命中映射”的基因名（不去重，保持顺序）
    gene_list = [rec["Approved_symbol"] for rec in mapping_records if rec["Approved_symbol"]]

    if gene_list:
        print(f"✅ 映射成功：用于富集的基因名数量={len(gene_list)}（原始条目={len(raw_gene_list)}）")
    else:
        print("⚠️ 映射后无可用于富集的基因名。将仅保存映射结果。")
        df_mapping["Classification ID"] = ""
        df_mapping["Cluster_Top_Terms"] = ""
        os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
        df_mapping.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
        print(f"✅ 已保存映射结果到：{OUTPUT_CSV_PATH}")
        raise SystemExit

except Exception as e:
    print(f"⚠️ ID->基因名映射失败：{e}。")
    # 仍然保存原始ID信息，分类设为 -1
    df_mapping = pd.DataFrame({
        "原始ID": raw_gene_list,
        "归一化ENSG": "",
        "Approved_symbol": "",
        "是否命中映射": False,
        "Classification ID": -1,
        "Cluster_Top_Terms": ""
    })
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_mapping.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
    print(f"✅ 已保存到：{OUTPUT_CSV_PATH}")
    raise SystemExit

print("-" * 50)
print(f"--- 开始富集分析（基因名数：{len(gene_list)}） ---")
print("-" * 50)

# -------------------------------------------------------------------------------------
# 3) GO 富集分析与显著结果筛选
#    若“已命中映射但无法富集（显著为空）”，则 Classification ID 设为 -1 并保存
# -------------------------------------------------------------------------------------
go_results = gp.enrichr(
    gene_list=gene_list,
    gene_sets='GO_Biological_Process_2023',
    organism='Human',
    outdir=None,
    no_plot=True 
)
df_enrichment = go_results.results
df_significant = df_enrichment[
    df_enrichment['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD
].sort_values(by='Adjusted P-value')

if df_significant.empty:
    print("⚠️ 无显著富集：已命中映射但无法富集，Classification ID 设为 -1 并保存。")
    df_out = df_mapping.copy()
    # 仅对“命中映射”的基因设为 -1；未命中映射的保持空以示区分
    df_out["Classification ID"] = df_out["是否命中映射"].apply(lambda x: -1 if x else "")
    df_out["Cluster_Top_Terms"] = ""
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
    print(f"✅ 已保存到：{OUTPUT_CSV_PATH}")
    raise SystemExit

# 构建基因-功能矩阵 (X)
all_genes = sorted(list(set(g for term_genes in df_significant['Genes'] for g in term_genes.split(';'))))
all_terms = df_significant['Term'].tolist()

gene_term_matrix = pd.DataFrame(0, index=all_genes, columns=all_terms)
for term in all_terms:
    genes_in_term = df_significant[df_significant['Term'] == term]['Genes'].iloc[0].split(';')
    gene_term_matrix.loc[genes_in_term, term] = 1
X = gene_term_matrix.values

# -------------------------------------------------------------------------------------
# 4) K-Means 聚类
#    若“可聚类基因数 < K”，则 Classification ID 设为 -1 并保存
# -------------------------------------------------------------------------------------
if len(all_genes) < K_CLUSTERS:
    print(f"⚠️ 基因数量 ({len(all_genes)}) 小于 K={K_CLUSTERS}。将 Classification ID 设为 -1 并保存。")
    df_out = df_mapping.copy()
    df_out["Classification ID"] = df_out["是否命中映射"].apply(lambda x: -1 if x else "")
    df_out["Cluster_Top_Terms"] = ""
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
    print(f"✅ 已保存到：{OUTPUT_CSV_PATH}")
    raise SystemExit

kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X)

df_classification = pd.DataFrame({
    'Gene Name': all_genes,
    'Classification ID': cluster_labels
}).sort_values(by='Classification ID')

# -------------------------------------------------------------------------------------
# 5) 计算每个分类组的Top术语，并加入分类表
# -------------------------------------------------------------------------------------
classification_reasons = {}
for i in range(K_CLUSTERS):
    genes_in_class = df_classification[df_classification['Classification ID'] == i]['Gene Name'].tolist()
    cluster_reason_terms = characterize_cluster(genes_in_class, i)
    classification_reasons[i] = " | ".join(cluster_reason_terms) if cluster_reason_terms else ""

df_classification["Cluster_Top_Terms"] = df_classification["Classification ID"].map(classification_reasons)

# -------------------------------------------------------------------------------------
# 6) 合并“映射结果”和“分类结果”，保存到一个CSV
# -------------------------------------------------------------------------------------
# 映射表以 Approved_symbol（基因名）与分类表的 Gene Name 对齐
df_out = df_mapping.merge(
    df_classification,
    left_on="Approved_symbol",
    right_on="Gene Name",
    how="left"
).drop(columns=["Gene Name"], errors='ignore')

# 保存CSV
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"✅ 已将映射与分类结果保存到：{OUTPUT_CSV_PATH}")

--- 初始基因条目数：250 ---
✅ 映射成功：用于富集的基因名数量=250（原始条目=250）
--------------------------------------------------
--- 开始富集分析（基因名数：250） ---
--------------------------------------------------


✅ 已将映射与分类结果保存到：select_genes/STNet_mapping.csv


In [9]:
import gseapy as gp
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

# --- 【重要参数设置】 ---
NPY_FILE_PATH = 'select_genes/HER2_Selected_Genes.npy'
OUTPUT_CSV_PATH = 'select_genes/HER2_mapping.csv'
K_CLUSTERS = 10                  # 设定的互斥分组数量
SIGNIFICANCE_THRESHOLD = 0.01    # GO 富集分析的 P 值阈值
TOP_TERMS_TO_REPORT = 3          # 每个分类组输出的最具代表性的功能数量

# --- 辅助函数：对单个基因分类组进行功能富集，以找到分类原因 ---
def characterize_cluster(gene_list, gene_set_name='GO_Biological_Process_2023'):
    """针对一个分类组的基因运行富集，返回Top术语列表"""
    if not gene_list:
        return []
    try:
        char_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set_name,
            organism='Human',
            outdir=None,
            no_plot=True 
        )
        df_char = char_results.results
        df_sig = df_char[df_char['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD]
        return df_sig.head(TOP_TERMS_TO_REPORT)['Term'].tolist()
    except Exception as e:
        return [f"分析失败 ({e})"]

# -------------------------------------------------------------------------------------
# 1) 从 .npy 文件中读取基因名（不做映射）
# -------------------------------------------------------------------------------------
try:
    gene_array = np.load(NPY_FILE_PATH, allow_pickle=True)
    gene_list_raw = [str(g).strip() for g in gene_array.flatten() if str(g).strip()]
except Exception as e:
    raise RuntimeError(f"读取 NPY 失败：{e}")

# 去重但保持顺序，构造基础输出表
seen = set()
base_genes = []
for g in gene_list_raw:
    if g not in seen:
        seen.add(g)
        base_genes.append(g)
base_df = pd.DataFrame({'Gene Name': base_genes})

print(f"--- 开始分析 {len(base_genes)} 个基因 ---")

# -------------------------------------------------------------------------------------
# 2) 主富集分析与显著项筛选
# -------------------------------------------------------------------------------------
go_results = gp.enrichr(
    gene_list=base_genes,
    gene_sets='GO_Biological_Process_2023',
    organism='Human',
    outdir=None,
    no_plot=True 
)
df_enrichment = go_results.results
df_significant = df_enrichment[
    df_enrichment['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD
].sort_values(by='Adjusted P-value')

# 若无显著富集：全部标记为 -1 并保存
if df_significant.empty:
    print("⚠️ 无显著富集结果，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# -------------------------------------------------------------------------------------
# 3) 构建基因-功能矩阵
# -------------------------------------------------------------------------------------
all_genes = sorted(list(set(g for term_genes in df_significant['Genes'] for g in term_genes.split(';'))))
all_terms = df_significant['Term'].tolist()

gene_term_matrix = pd.DataFrame(0, index=all_genes, columns=all_terms)
for term in all_terms:
    genes_in_term = df_significant.loc[df_significant['Term'] == term, 'Genes'].iloc[0].split(';')
    # 仅在索引存在的基因上置1，避免越界
    genes_in_term = [g for g in genes_in_term if g in gene_term_matrix.index]
    gene_term_matrix.loc[genes_in_term, term] = 1

X = gene_term_matrix.values

# 若基因数量不足以聚类：全部标记为 -1 并保存
if len(all_genes) < K_CLUSTERS:
    print(f"⚠️ 可用于聚类的基因数({len(all_genes)}) < K({K_CLUSTERS})，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# -------------------------------------------------------------------------------------
# 4) K-Means 硬性聚类
# -------------------------------------------------------------------------------------
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X)

df_classification = pd.DataFrame({
    'Gene Name': all_genes,
    'Classification ID': cluster_labels
}).sort_values(by='Classification ID')

# -------------------------------------------------------------------------------------
# 5) 为每个分类组计算Top术语，并合并到每个基因
# -------------------------------------------------------------------------------------
classification_reasons = {}
for cid in range(K_CLUSTERS):
    genes_in_class = df_classification[df_classification['Classification ID'] == cid]['Gene Name'].tolist()
    top_terms = characterize_cluster(genes_in_class)
    classification_reasons[cid] = " | ".join(top_terms) if top_terms else ""

df_classification['Cluster_Top_Terms'] = df_classification['Classification ID'].map(classification_reasons)

# -------------------------------------------------------------------------------------
# 6) 与基础表合并并保存为CSV（未聚到类的基因设为 -1）
# -------------------------------------------------------------------------------------
df_out = base_df.merge(df_classification, on='Gene Name', how='left')
df_out['Classification ID'] = df_out['Classification ID'].fillna(-1).astype(int)
df_out['Cluster_Top_Terms'] = df_out['Cluster_Top_Terms'].fillna('')

os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✅ 已保存分类结果到：{OUTPUT_CSV_PATH}")

--- 开始分析 250 个基因 ---


✅ 已保存分类结果到：select_genes/HER2_mapping.csv


In [22]:
import gseapy as gp
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

# --- 【重要参数设置】 ---
NPY_FILE_PATH = 'select_genes/HEST_MOUSE_BRAIN_gene.npy'
OUTPUT_CSV_PATH = 'select_genes/HEST_MOUSE_BRAIN_mapping.csv'
K_CLUSTERS = 10                  # 设定的互斥分组数量
SIGNIFICANCE_THRESHOLD = 0.05    # GO 富集分析的 P 值阈值
TOP_TERMS_TO_REPORT = 3          # 每个分类组输出的最具代表性的功能数量

# --- 辅助函数：对单个基因分类组进行功能富集，以找到分类原因 ---
def characterize_cluster(gene_list, gene_set_name='KEGG_2019_Mouse'):
    """针对某个分类组的基因运行富集分析，返回Top术语列表"""
    if not gene_list:
        return []
    try:
        char_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set_name,
            organism='Mouse',
            outdir=None,
            no_plot=True 
        )
        df_char = char_results.results
        df_sig = df_char[df_char['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD]
        return df_sig.head(TOP_TERMS_TO_REPORT)['Term'].tolist()
    except Exception as e:
        return [f"分析失败 ({e})"]

# -------------------------------------------------------------------------------------
# 1) 从 .npy 文件中读取基因名（不做映射），去重但保持原顺序
# -------------------------------------------------------------------------------------
try:
    gene_array = np.load(NPY_FILE_PATH, allow_pickle=True)
    gene_list_raw = [str(g).strip() for g in gene_array.flatten() if str(g).strip()]
except Exception as e:
    raise RuntimeError(f"读取 NPY 失败：{e}")

seen = set()
base_genes = []
for g in gene_list_raw:
    if g not in seen:
        seen.add(g)
        base_genes.append(g)

base_df = pd.DataFrame({'Gene Name': base_genes})
print(f"--- 开始分析 {len(base_genes)} 个基因 ---")

# -------------------------------------------------------------------------------------
# 2) 主富集分析与显著项筛选
# -------------------------------------------------------------------------------------
go_results = gp.enrichr(
    gene_list=base_genes,
    gene_sets='KEGG_2019_Mouse',
    organism='Mouse',
    outdir=None,
    no_plot=True 
)
df_enrichment = go_results.results
df_significant = df_enrichment[
    df_enrichment['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD
].sort_values(by='Adjusted P-value')

# 若无显著富集：全部标记为 -1 并保存
if df_significant.empty:
    print("⚠️ 无显著富集结果，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# -------------------------------------------------------------------------------------
# 3) 构建基因-功能矩阵
# -------------------------------------------------------------------------------------
all_genes = sorted(list(set(g for term_genes in df_significant['Genes'] for g in term_genes.split(';'))))
all_terms = df_significant['Term'].tolist()

gene_term_matrix = pd.DataFrame(0, index=all_genes, columns=all_terms)
for term in all_terms:
    genes_in_term = df_significant.loc[df_significant['Term'] == term, 'Genes'].iloc[0].split(';')
    genes_in_term = [g for g in genes_in_term if g in gene_term_matrix.index]  # 仅对存在于索引的基因置1
    gene_term_matrix.loc[genes_in_term, term] = 1

X = gene_term_matrix.values

# 若基因数量不足以聚类：全部标记为 -1 并保存
if len(all_genes) < K_CLUSTERS:
    print(f"⚠️ 可用于聚类的基因数({len(all_genes)}) < K({K_CLUSTERS})，全部标记为 -1。")
    df_out = base_df.copy()
    df_out['Classification ID'] = -1
    df_out['Cluster_Top_Terms'] = ''
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"已保存：{OUTPUT_CSV_PATH}")
    raise SystemExit

# -------------------------------------------------------------------------------------
# 4) K-Means 聚类
# -------------------------------------------------------------------------------------
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X)

df_classification = pd.DataFrame({
    'Gene Name': all_genes,
    'Classification ID': cluster_labels
}).sort_values(by='Classification ID')

# -------------------------------------------------------------------------------------
# 5) 为每个分类组计算Top术语，并合并到每个基因
# -------------------------------------------------------------------------------------
classification_reasons = {}
for cid in range(K_CLUSTERS):
    genes_in_class = df_classification[df_classification['Classification ID'] == cid]['Gene Name'].tolist()
    top_terms = characterize_cluster(genes_in_class)
    classification_reasons[cid] = " | ".join(top_terms) if top_terms else ""

df_classification['Cluster_Top_Terms'] = df_classification['Classification ID'].map(classification_reasons)

# -------------------------------------------------------------------------------------
# 6) 与基础基因表合并并保存为CSV（未聚到类的基因设为 -1）
# -------------------------------------------------------------------------------------
df_out = base_df.merge(df_classification, on='Gene Name', how='left')
df_out['Classification ID'] = df_out['Classification ID'].fillna(-1).astype(int)
df_out['Cluster_Top_Terms'] = df_out['Cluster_Top_Terms'].fillna('')

os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✅ 已保存分类结果到：{OUTPUT_CSV_PATH}")

--- 开始分析 250 个基因 ---


✅ 已保存分类结果到：select_genes/HEST_MOUSE_BRAIN_mapping.csv


In [27]:
import gseapy as gp
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os
import sys

# --- 【重要参数设置】 ---
NPY_FILE_PATH = 'select_genes/HEST_MOUSE_BRAIN_gene.npy'
OUTPUT_CSV_PATH = 'select_genes/HEST_MOUSE_BRAIN_mapping.csv'
K_CLUSTERS = 10 # 设定的互斥分组数量
SIGNIFICANCE_THRESHOLD = 0.05 # GO 富集分析的 P 值阈值
TOP_TERMS_TO_REPORT = 3 # 每个分类组输出的最具代表性的功能数量
GENE_SETS_TO_USE = ['KEGG_2019_Mouse', 'GO_Biological_Process_2023'] # 优化：使用多个基因集

# --- 辅助函数：对单个基因分类组进行功能富集，以找到分类原因 ---
def characterize_cluster(gene_list, gene_set_name='KEGG_2019_Mouse'):
    """针对某个分类组的基因运行富集分析，返回Top术语列表"""
    if not gene_list:
        return []
    try:
        char_results = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_set_name,
            organism='Mouse',
            outdir=None,
            no_plot=True 
        )
        df_char = char_results.results
        df_sig = df_char[df_char['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD]
        df_sig = df_sig.sort_values(by=['Adjusted P-value', 'Overlap'], ascending=[True, False])

        return df_sig.head(TOP_TERMS_TO_REPORT)['Term'].tolist()
    except Exception as e:
        return [f"分析失败 ({e})"]

# -------------------------------------------------------------------------------------
# 1) 从 .npy 文件中读取基因名，去重
# -------------------------------------------------------------------------------------
print(f"--- 正在从 {NPY_FILE_PATH} 读取基因 ---")
try:
    gene_array = np.load(NPY_FILE_PATH, allow_pickle=True)
    gene_list_raw = [str(g).strip() for g in gene_array.flatten() if str(g).strip()]
except Exception as e:
    print(f"致命错误：读取 NPY 失败：{e}", file=sys.stderr)
    sys.exit(1)

seen = set()
base_genes = []
for g in gene_list_raw:
    if g not in seen:
        seen.add(g)
        base_genes.append(g)

if not base_genes:
    print("致命错误：基因列表为空！", file=sys.stderr)
    sys.exit(1)

base_df = pd.DataFrame({'Gene Name': base_genes})
print(f"--- 开始分析 {len(base_genes)} 个基因 ---")

# -------------------------------------------------------------------------------------
# 2) 主富集分析与显著项筛选
# -------------------------------------------------------------------------------------
print(f"--- 运行富集分析 ({' & '.join(GENE_SETS_TO_USE)}) ---")
go_results = gp.enrichr(
    gene_list=base_genes,
    gene_sets=GENE_SETS_TO_USE,
    organism='Mouse',
    outdir=None,
    no_plot=True 
)
df_enrichment = go_results.results

df_significant = df_enrichment[
    df_enrichment['Adjusted P-value'] < SIGNIFICANCE_THRESHOLD
].sort_values(by=['Adjusted P-value', 'Overlap'], ascending=[True, False])

if df_significant.empty:
    # 调试信息和退出逻辑（保持不变）...
    # ...
    sys.exit(0)

# -------------------------------------------------------------------------------------
# 3) 构建基因-功能矩阵
# -------------------------------------------------------------------------------------
all_genes = sorted(list(set(g for term_genes in df_significant['Genes'] for g in term_genes.split(';'))))

print(f"--- 调试信息：参与聚类的基因总数 (all_genes)：{len(all_genes)} 个 ---")

all_terms = df_significant['Term'].tolist()

gene_term_matrix = pd.DataFrame(0, index=all_genes, columns=all_terms)
for term in all_terms:
    genes_in_term = df_significant.loc[df_significant['Term'] == term, 'Genes'].iloc[0].split(';')
    genes_in_term = [g for g in genes_in_term if g in gene_term_matrix.index]
    gene_term_matrix.loc[genes_in_term, term] = 1

X = gene_term_matrix.values

if len(all_genes) < K_CLUSTERS:
    # 警告和退出逻辑（保持不变）...
    # ...
    sys.exit(0)

# -------------------------------------------------------------------------------------
# 4) K-Means 聚类
# -------------------------------------------------------------------------------------
print(f"--- 运行 K-Means 聚类 (K={K_CLUSTERS}) ---")
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X)

df_classification = pd.DataFrame({
    'Gene Name': all_genes,
    'Classification ID': cluster_labels
}).sort_values(by='Classification ID')

# -------------------------------------------------------------------------------------
# 5) 为每个分类组计算Top术语
# -------------------------------------------------------------------------------------
classification_reasons = {}
for cid in range(K_CLUSTERS):
    genes_in_class = df_classification[df_classification['Classification ID'] == cid]['Gene Name'].tolist()
    top_terms = characterize_cluster(genes_in_class, gene_set_name='KEGG_2019_Mouse') 
    classification_reasons[cid] = " | ".join(top_terms) if top_terms else ""

df_classification['Cluster_Top_Terms'] = df_classification['Classification ID'].map(classification_reasons)

# -------------------------------------------------------------------------------------
# 6) 【关键修改点】：标准化基因名称后再合并
# -------------------------------------------------------------------------------------
print("--- 正在标准化基因名称并合并结果... ---")

# 1. 备份原始基因名
base_df['Original_Gene_Name'] = base_df['Gene Name']
df_classification['Original_Gene_Name'] = df_classification['Gene Name']

# 2. 标准化用于合并的列：去除空格并转大写
base_df['Gene Name'] = base_df['Gene Name'].str.strip().str.upper()
df_classification['Gene Name'] = df_classification['Gene Name'].str.strip().str.upper()

# 3. 合并
df_out = base_df.merge(
    df_classification[['Gene Name', 'Classification ID', 'Cluster_Top_Terms']], 
    on='Gene Name', 
    how='left'
)

# 4. 恢复原始基因名并清理
df_out['Gene Name'] = df_out['Original_Gene_Name']
df_out = df_out.drop(columns=['Original_Gene_Name'])


# 5. 填充未分类基因并保存
df_out['Classification ID'] = df_out['Classification ID'].fillna(-1).astype(int)
df_out['Cluster_Top_Terms'] = df_out['Cluster_Top_Terms'].fillna('')


os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
df_out.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"✅ 成功完成分析！已保存分类结果到：{OUTPUT_CSV_PATH}")

--- 正在从 select_genes/HEST_MOUSE_BRAIN_gene.npy 读取基因 ---
--- 开始分析 250 个基因 ---
--- 运行富集分析 (KEGG_2019_Mouse & GO_Biological_Process_2023) ---


--- 调试信息：参与聚类的基因总数 (all_genes)：167 个 ---
--- 运行 K-Means 聚类 (K=10) ---
--- 正在标准化基因名称并合并结果... ---
✅ 成功完成分析！已保存分类结果到：select_genes/HEST_MOUSE_BRAIN_mapping.csv
